#### Preparation

In [2]:
%pip install -U scikit-learn

   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ------- -------------------------------- 1.6/8.0 MB 13.4 MB/s eta 0:00:01
   ---------------------------------------- 8.0/8.0 MB 28.5 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.3 requires scikit-learn<1.7,>=1.2, but you have scikit-learn 1.8.0 which is incompatible.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.models import resnet50, ResNet50_Weights
from torch.utils.data import DataLoader
import os
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm import tqdm

In [ ]:
DATA_DIR = r"C:\Users\e6va6je238\Desktop\Cassava_Leaf_Datasets\Classification"
WEIGHTS_DIR = "../weights"
BATCH_SIZE = 32
NUM_EPOCHS_FREEZE = 8
NUM_EPOCHS_FINETUNE = 20
LR = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [3]:
weights = ResNet50_Weights.DEFAULT

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean,
                         std=std)
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean,
                         std=std)
])


In [4]:
train_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "train"),
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "val"),
    transform=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

class_names = train_dataset.classes
num_classes = len(class_names)

print(class_names)


['Healthy', 'Spider Mites', 'leaf blight', 'leaf spot']


In [5]:
model = resnet50(weights=weights)

In [6]:
# Freeze all layers
for param in model.parameters():
    param.requires_grad = False


In [7]:
# Replace final layer for dataset classes
in_features = model.fc.in_features

model.fc = nn.Linear(in_features, num_classes)

In [8]:
model = model.to(DEVICE)

In [9]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.fc.parameters(),
    lr=LR
)

In [10]:
def train_one_epoch(loader, epoch, epochs):
    model.train()
    
    train_loss = 0
    train_preds = []
    train_labels = []

    for images, labels in tqdm(loader, desc=f"Training Epoch {epoch+1}/{epochs}"):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds = outputs.argmax(dim=1)

        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

        train_acc = accuracy_score(train_labels, train_preds)


    return train_loss, train_acc


In [11]:
def validate(loader):
    model.eval()
    
    val_loss = 0
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            preds = outputs.argmax(dim=1)

            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
        
        val_acc = accuracy_score(val_labels, val_preds)

        precision, recall, f1, _ = precision_recall_fscore_support(
        val_labels, val_preds, average='weighted'
        )   

    return precision, recall, f1, val_loss, val_acc

#### Training 1 (Freeze Backbone)

In [12]:

print("\n-----------Starting Phase 1 Training-----------\n")

for epoch in range(NUM_EPOCHS_FREEZE):
    train_loss, train_acc = train_one_epoch(train_loader, epoch, NUM_EPOCHS_FREEZE)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS_FREEZE}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")




-----------Starting Phase 1 Training-----------



Training Epoch 1/8: 100%|██████████| 30/30 [05:39<00:00, 11.32s/it]



Epoch [1/8]
Train Loss: 30.3813 | Train Acc: 0.6813
Val Loss: 6.9552 | Val Acc: 0.8375
Precision: 0.8494 | Recall: 0.8375 | F1: 0.8353


Training Epoch 2/8: 100%|██████████| 30/30 [05:26<00:00, 10.90s/it]



Epoch [2/8]
Train Loss: 17.4458 | Train Acc: 0.8667
Val Loss: 5.0990 | Val Acc: 0.8542
Precision: 0.8597 | Recall: 0.8542 | F1: 0.8543


...


Epoch [6/8]
Train Loss: 9.6886 | Train Acc: 0.9146
Val Loss: 3.6306 | Val Acc: 0.8792
Precision: 0.8790 | Recall: 0.8792 | F1: 0.8791


Training Epoch 7/8: 100%|██████████| 30/30 [05:25<00:00, 10.83s/it]



Epoch [7/8]
Train Loss: 9.2648 | Train Acc: 0.9104
Val Loss: 3.3377 | Val Acc: 0.8875
Precision: 0.8872 | Recall: 0.8875 | F1: 0.8873


Training Epoch 8/8: 100%|██████████| 30/30 [05:27<00:00, 10.93s/it]



Epoch [8/8]
Train Loss: 8.3800 | Train Acc: 0.9260
Val Loss: 3.2364 | Val Acc: 0.8917
Precision: 0.8915 | Recall: 0.8917 | F1: 0.8916


#### Training 2 Fine-Tune

In [13]:
for param in model.layer4.parameters():
    param.requires_grad = True

In [ ]:
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5
)

In [ ]:
best_val_acc = 0.0

print("\n-----------Starting Fine-tuning Stage-----------\n")

for epoch in range(NUM_EPOCHS_FINETUNE):
    train_loss, train_acc = train_one_epoch(train_loader, epoch, NUM_EPOCHS_FINETUNE)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS_FINETUNE}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")


torch.save({
            "model_state_dict": model.state_dict(),
            "class_names": class_names
        }, os.path.join(WEIGHTS_DIR, "ResNet50.pth"))


-----------Starting Fine-tuning Stage-----------



Training Epoch 1/20: 100%|██████████| 30/30 [05:39<00:00, 11.31s/it]



Epoch [1/20]
Train Loss: 6.9840 | Train Acc: 0.9198
Val Loss: 2.4239 | Val Acc: 0.9333
Precision: 0.9362 | Recall: 0.9333 | F1: 0.9337


Training Epoch 2/20: 100%|██████████| 30/30 [17:13<00:00, 34.45s/it]



Epoch [2/20]
Train Loss: 3.9676 | Train Acc: 0.9552
Val Loss: 1.8971 | Val Acc: 0.9500
Precision: 0.9518 | Recall: 0.9500 | F1: 0.9502


...


Epoch [18/20]
Train Loss: 0.2774 | Train Acc: 0.9979
Val Loss: 1.1411 | Val Acc: 0.9708
Precision: 0.9718 | Recall: 0.9708 | F1: 0.9707


Training Epoch 19/20: 100%|██████████| 30/30 [17:29<00:00, 34.98s/it]



Epoch [19/20]
Train Loss: 0.2193 | Train Acc: 0.9990
Val Loss: 1.0963 | Val Acc: 0.9750
Precision: 0.9751 | Recall: 0.9750 | F1: 0.9750


Training Epoch 20/20: 100%|██████████| 30/30 [19:06<00:00, 38.21s/it]



Epoch [20/20]
Train Loss: 0.1830 | Train Acc: 0.9979
Val Loss: 1.0561 | Val Acc: 0.9708
Precision: 0.9711 | Recall: 0.9708 | F1: 0.9708


#### Predicting Sample

In [19]:
from PIL import Image

checkpoint = torch.load("../weights/ResNet50.pth")

model.load_state_dict(checkpoint["model_state_dict"])
class_names = checkpoint["class_names"]

model.eval()

def predict(image_path):
    image = Image.open(image_path).convert("RGB")
    image = val_transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs, 1)

    return class_names[pred.item()]


print(predict(r"C:\Users\e6va6je238\Desktop\Cassava_Leaf_Datasets\Classification\val\leaf blight\leaf blight_val_21.jpg"))

leaf blight
